In [5]:
# =========================
# STAGES 1–6 END-TO-END PIPELINE
# (01 Scoping → 02 Tooling → 03 Python → 04 Ingestion → 05 Storage → 06 Preprocessing)
# =========================

from __future__ import annotations
import os, io, math, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# ---------- Optional deps ----------
try:
    import yfinance as yf
except Exception:
    yf = None

try:
    import pyarrow  # noqa: F401
    PARQUET_OK = True
except Exception:
    PARQUET_OK = False


# =========================
# STAGE 02 — ENV & PATHS
# =========================

def load_env_simple(path: str | Path = ".env") -> None:
    """
    Minimal .env loader (KEY=VALUE). No external dependency.
    Only sets keys not already in os.environ.
    """
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        k, v = k.strip(), v.strip()
        if k and k not in os.environ:
            os.environ[k] = v

load_env_simple()

# Use your explicit Windows paths; allow env overrides if present
RAW = Path(os.getenv(
    "DATA_DIR_RAW",
    r"C:\Users\User\bootcamp_Khushi_Khanna\project\revenue_risk_model\data\raw"
))
PRO = Path(os.getenv(
    "DATA_DIR_PROCESSED",
    r"C:\Users\User\bootcamp_Khushi_Khanna\project\revenue_risk_model\data\processed"
))

for d in (RAW, PRO):
    d.mkdir(parents=True, exist_ok=True)

def nowstamp() -> str:
    return datetime.now().strftime("%Y%m%d-%H%M")


# =========================
# STAGE 03 — PYTHON FUNDAMENTALS (tiny)
# =========================

arr = np.array([1, 2, 3, 4]) * 10
print("Vectorized multiply:", arr)  # -> [10 20 30 40]


# =========================
# STAGE 04 — DATA INGESTION
# =========================

def api_try_yfinance_jpm() -> pd.DataFrame | None:
    """
    Attempt a tiny yfinance pull for JPM quarterly financials.
    Produces a df with ['date','revenue','source','ticker'] or returns None.
    """
    if yf is None:
        return None
    try:
        t = yf.Ticker("JPM")
        qfin = t.quarterly_financials
        if qfin is None or qfin.empty:
            return None

        # Prefer a revenue-like row
        row = None
        for label in ["Total Revenue", "Total Revenue As Reported", "Operating Revenue", "Revenue"]:
            if label in qfin.index:
                row = label
                break
        if row is None:
            # fallback: any 'revenue' row
            revish = [idx for idx in qfin.index if "revenue" in idx.lower()]
            if revish:
                row = revish[0]
            else:
                # last fallback: sum positive rows per column
                s = qfin[qfin > 0].sum()
                df = pd.DataFrame({"date": s.index, "revenue": s.values})
                df["source"] = "yfinance"
                df["ticker"] = "JPM"
                return df

        series = qfin.loc[row]
        df = pd.DataFrame({"date": series.index, "revenue": series.values})
        df["source"] = "yfinance"
        df["ticker"] = "JPM"
        return df
    except Exception as e:
        print("yfinance pull failed:", repr(e))
        return None

def api_synthetic_jpm() -> pd.DataFrame:
    """
    Offline-friendly fallback with 8 quarterly points.
    """
    dates = pd.period_range("2023Q1", periods=8, freq="Q").to_timestamp("Q")
    rev = np.array([42.1, 44.0, 43.2, 46.5, 48.0, 47.5, 49.1, 50.2]) * 1e9
    df = pd.DataFrame({"date": dates, "revenue": rev})
    df["source"] = "synthetic"
    df["ticker"] = "JPM"
    return df

def api_coerce_and_clean(df: pd.DataFrame | None) -> pd.DataFrame | None:
    """
    Ensure 'date' and 'revenue' exist; coerce to proper types; drop rows with NA.
    Return None if unusable after cleaning.
    """
    if df is None or df.empty or not {"date", "revenue"}.issubset(df.columns):
        return None
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["revenue"] = pd.to_numeric(out["revenue"], errors="coerce")
    n0 = len(out)
    out = out.dropna(subset=["date", "revenue"]).sort_values("date").reset_index(drop=True)
    dropped = n0 - len(out)
    if dropped > 0:
        print(f"[API] Dropped {dropped} rows with NA date/revenue during validation.")
    return None if out.empty else out

# Pull → Clean → Fallback if required
_df_api_raw = api_try_yfinance_jpm()
df_api = api_coerce_and_clean(_df_api_raw)
if df_api is None:
    print("API df unusable; switching to synthetic fallback.")
    df_api = api_coerce_and_clean(api_synthetic_jpm())

# Save raw API CSV (timestamped)
api_raw_path = RAW / f"api_yf_JPM_{nowstamp()}.csv"
df_api.to_csv(api_raw_path, index=False)
print(f"Saved raw API -> {api_raw_path}")

# ---- Scrape (with robust fallback) ----
def scrape_try_simple() -> pd.DataFrame | None:
    """
    Try to read a simple wikitable; uses a UA header and pandas.read_html.
    Returns a small table or None if blocked/shape mismatch.
    """
    import urllib.request
    url = "https://en.wikipedia.org/wiki/List_of_largest_banks"
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/122.0.0.0 Safari/537.36")
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            html = resp.read()
        tables = pd.read_html(io.BytesIO(html))
        # Pick a reasonable table
        cands = [t for t in tables if t.shape[1] >= 3 and len(t) >= 3]
        if not cands:
            return None
        df = cands[0].copy().iloc[:, :3]
        df.columns = [f"col_{i}" for i in range(df.shape[1])]
        return df
    except Exception as e:
        print("read_html failed:", repr(e))
        return None

def scrape_synthetic() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "bank": ["JPMorgan", "Bank of America", "ICBC"],
            "country": ["USA", "USA", "China"],
            "assets_usd_bil": [3900, 3200, 5600],
        }
    )

df_scrape = scrape_try_simple()
if df_scrape is None or df_scrape.empty:
    print("No suitable HTML table found; switching to synthetic fallback.")
    df_scrape = scrape_synthetic()

scrape_raw_path = RAW / f"scrape_top_banks_{nowstamp()}.csv"
df_scrape.to_csv(scrape_raw_path, index=False)
print(f"Saved raw scrape -> {scrape_raw_path}")


# =========================
# STAGE 05 — STORAGE (CSV + PARQUET) + VALIDATION
# =========================

def write_df(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    suf = path.suffix.lower()
    if suf == ".csv":
        df.to_csv(path, index=False)
    elif suf == ".parquet":
        if not PARQUET_OK:
            raise RuntimeError("Parquet engine missing. Install pyarrow or use CSV.")
        df.to_parquet(path, index=False)
    else:
        raise ValueError(f"Unsupported extension: {suf}")
    print(f"Wrote {path}")
    return path

def read_df(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    elif suf == ".parquet":
        if not PARQUET_OK:
            raise RuntimeError("Parquet engine missing. Install pyarrow or open CSV.")
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported extension: {suf}")

# Save the API df to processed as CSV and (if available) Parquet
api_pro_csv = PRO / f"api_yf_JPM_{nowstamp()}.csv"
api_pro_parq = PRO / f"api_yf_JPM_{nowstamp()}.parquet"

write_df(df_api, api_pro_csv)
if PARQUET_OK:
    write_df(df_api, api_pro_parq)

# Reload to validate round-trip
re_csv = read_df(api_pro_csv)
print("Reloaded CSV shape:", re_csv.shape)
if PARQUET_OK:
    re_parq = read_df(api_pro_parq)
    print("Reloaded Parquet shape:", re_parq.shape)
    assert re_csv.shape == re_parq.shape, "CSV vs Parquet shape mismatch"


# =========================
# STAGE 06 — PREPROCESSING (CLEANING)
# =========================

def fill_missing_median(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Fill NAs in 'cols' with the column median (numeric coercion).
    """
    out = df.copy()
    for c in cols:
        if c in out:
            x = pd.to_numeric(out[c], errors="coerce")
            med = x.median()
            out[c] = x.fillna(med)
    return out

def drop_missing(df: pd.DataFrame, cols: list[str], thresh: float = 0.9) -> pd.DataFrame:
    """
    Keep rows where >= ceil(len(cols)*thresh) of the selected columns are present.
    """
    out = df.copy()
    sel = out[cols].copy()
    keep = sel.notna().sum(axis=1) >= math.ceil(len(cols) * thresh)
    return out.loc[keep].reset_index(drop=True)

def normalize_data(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Z-score normalize numeric cols (mean 0, std 1). Constant columns → 0.
    """
    out = df.copy()
    for c in cols:
        if c in out:
            x = pd.to_numeric(out[c], errors="coerce")
            mu, sd = x.mean(), x.std(ddof=0)
            out[c] = (x - mu) / sd if (sd and not np.isnan(sd)) else 0.0
    return out

# Build a tiny risk proxy to demo cleaning on 2 numeric columns
df_work = df_api.copy()
df_work["risk_index_var95"] = (
    pd.Series(df_work["revenue"])
      .pct_change()
      .abs()
      .rolling(3, min_periods=1)
      .mean()
)

num_cols = ["revenue", "risk_index_var95"]

# Cleaning pipeline
df_step1 = fill_missing_median(df_work, num_cols)
df_step2 = drop_missing(df_step1, num_cols, thresh=0.9)
df_clean = normalize_data(df_step2, num_cols)

clean_out = PRO / f"clean_api_yf_JPM_{nowstamp()}.csv"
write_df(df_clean, clean_out)

print("\n=== CLEAN SAMPLE (top 5) ===")
print(df_clean.head())
print("\nDone ✅")

# --- keep originals before normalization
df_work = df_api.copy()

# risk proxy on ORIGINAL scale
df_work["risk_index_var95"] = (
    pd.Series(df_work["revenue"])
      .pct_change()
      .abs()
      .rolling(3, min_periods=1)
      .mean()
)

num_cols = ["revenue", "risk_index_var95"]

# cleaning on original cols
df_step1 = fill_missing_median(df_work, num_cols)
df_step2 = drop_missing(df_step1, num_cols, thresh=0.9)

# make z-scored copies so originals stay intact
df_with_z = df_step2.copy()
for c in num_cols:
    x = pd.to_numeric(df_with_z[c], errors="coerce")
    mu, sd = x.mean(), x.std(ddof=0)
    df_with_z[f"{c}_z"] = (x - mu) / sd if (sd and not np.isnan(sd)) else 0.0

clean_out = PRO / f"clean_api_yf_JPM_{nowstamp()}.csv"
write_df(df_with_z, clean_out)

print(df_with_z.head())


Vectorized multiply: [10 20 30 40]
[API] Dropped 2 rows with NA date/revenue during validation.
Saved raw API -> data\raw\api_yf_JPM_20250828-1740.csv
Saved raw scrape -> data\raw\scrape_top_banks_20250828-1740.csv
Wrote data\processed\api_yf_JPM_20250828-1740.csv
Wrote data\processed\api_yf_JPM_20250828-1740.parquet
Reloaded CSV shape: (5, 4)
Reloaded Parquet shape: (5, 4)
Wrote data\processed\clean_api_yf_JPM_20250828-1740.csv

=== CLEAN SAMPLE (top 5) ===
        date   revenue    source ticker  risk_index_var95
0 2024-06-30 -1.132424  yfinance    JPM          0.127536
1 2024-09-30 -0.681540  yfinance    JPM         -0.674001
2 2024-12-31 -0.578021  yfinance    JPM         -1.531656
3 2025-03-31  1.366607  yfinance    JPM          1.149047
4 2025-06-30  1.025377  yfinance    JPM          0.929074

Done ✅
Wrote data\processed\clean_api_yf_JPM_20250828-1740.csv
        date       revenue    source ticker  risk_index_var95  revenue_z  \
0 2024-06-30  4.206800e+10  yfinance    JPM      

In [20]:
# stages_07_to_10_pipeline.py
# Revenue–Risk Project: Stages 7 → 10 (Outliers, EDA, Feature Eng, LR, TS/Classification)
# - Auto-detect latest data/raw file (or by ticker)
# - Merge real risk series if available; else build proxy
# - Version-safe metrics; robust on tiny datasets
# - No deprecated pandas args; no squared= in sklearn

import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Try time-series libs; pipeline works even if they’re unavailable
try:
    import statsmodels.api as sm  # noqa: F401
    from statsmodels.tsa.arima.model import ARIMA
except Exception:
    ARIMA = None

warnings.filterwarnings("ignore")

# ---------------------------
# Settings
# ---------------------------
ALLOW_SMALL_N = True           # run LR even on very small samples (>=3)
TICKER_PREFERENCE = "JPM"      # choose None to disable ticker preference

# ---------------------------
# Globals & small helpers
# ---------------------------
NUMERIC_CANDIDATES = [
    "revenue", "risk_index_var95", "risk_score", "n_clients", "assets", "liabilities"
]
DATE_CANDIDATES = ["date", "Date", "asof_date", "timestamp"]

def rmse(y_true, y_pred) -> float:
    """Version-safe RMSE (no `squared=` kwarg)."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def smart_parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    for c in DATE_CANDIDATES:
        if c in df.columns:
            try:
                df[c] = pd.to_datetime(df[c], errors="coerce")
            except Exception:
                pass
    return df

def load_data(path: str) -> pd.DataFrame:
    """Load CSV/Parquet robustly; coerce numerics; sort by date if present."""
    assert os.path.exists(path), f"File not found: {path}"
    if path.lower().endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)  # no deprecated infer_datetime_format
    df = smart_parse_dates(df)
    for c in NUMERIC_CANDIDATES:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in DATE_CANDIDATES:
        if c in df.columns:
            df = df.sort_values(c)
            break
    return df.reset_index(drop=True)

# ---------------------------
# Risk loader / proxy
# ---------------------------
def _latest(paths):
    paths = list(paths)
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

def load_risk_series(data_dir="data/raw"):
    """
    Try to load a risk/VAR95 series from data/raw; return [date, risk_index_var95].
    Accepts flexible names: ['risk_index_var95','var95','VaR95','risk_index','risk_var','risk'].
    """
    p = Path(data_dir)
    cands = (
        list(p.glob("*var95*.csv")) + list(p.glob("*var95*.parquet")) +
        list(p.glob("*risk*var*.csv")) + list(p.glob("*risk*var*.parquet")) +
        list(p.glob("revrisk_var95*.csv")) + list(p.glob("revrisk_var95*.parquet"))
    )
    f = _latest(cands)
    if f is None:
        return None

    rdf = pd.read_parquet(f) if f.suffix.lower()==".parquet" else pd.read_csv(f)
    rdf = smart_parse_dates(rdf)

    # date column
    dcol = None
    for c in DATE_CANDIDATES:
        if c in rdf.columns:
            dcol = c; break
    if dcol is None:
        for c in rdf.columns:
            try:
                pd.to_datetime(rdf[c])
                dcol = c; break
            except Exception:
                pass
    if dcol is None:
        return None

    # risk column
    cand_names = ["risk_index_var95","var95","VaR95","risk_index","risk_var","risk"]
    rcol = None
    for name in cand_names:
        if name in rdf.columns:
            rcol = name; break
    if rcol is None:
        for c in rdf.columns:
            if c != dcol and pd.api.types.is_numeric_dtype(rdf[c]):
                rcol = c; break
    if rcol is None:
        return None

    out = rdf[[dcol, rcol]].dropna().copy()
    out.columns = ["date", "risk_index_var95"]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return out

def attach_risk_or_proxy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure df has 'risk_index_var95'.
    1) Try merging nearest-dated real series from data/raw (±5 days).
    2) For any missing rows (or if none found), fill with a proxy:
       rolling std of revenue pct change, scaled to ≈100 median.
    """
    result = df.copy()

    # Try to merge a real series
    if "date" in result.columns:
        risk_df = load_risk_series("data/raw")
        if risk_df is not None:
            merged = pd.merge_asof(
                result.sort_values("date"),
                risk_df.sort_values("date"),
                on="date",
                direction="nearest",
                tolerance=pd.Timedelta(days=5)
            )
            result = merged

    # Build proxy for any missing / absent risk values
    if "revenue" in result.columns:
        proxy = result["revenue"].pct_change().rolling(3, min_periods=2).std() * np.sqrt(4) * 100
        # scale proxy to ~100 median if feasible
        med = np.nanmedian(proxy)
        if np.isfinite(med) and med > 0:
            proxy = proxy * (100.0 / med)

        if "risk_index_var95" in result.columns:
            result["risk_index_var95"] = result["risk_index_var95"].fillna(proxy)
        else:
            result["risk_index_var95"] = proxy

    return result

# ---------------------------
# Stage 7: Outliers & Assumptions
# ---------------------------
@dataclass
class OutlierResults:
    iqr_bounds: Dict[str, Tuple[float, float]]
    n_outliers: Dict[str, int]
    df_no_outliers: pd.DataFrame
    df_winsorized: pd.DataFrame

def iqr_bounds(series: pd.Series, k: float = 1.5) -> Tuple[float, float]:
    q1, q3 = np.nanpercentile(series.dropna(), [25, 75])
    iqr = q3 - q1
    return (q1 - k * iqr, q3 + k * iqr)

def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99) -> pd.Series:
    lo, hi = s.quantile(lower_q), s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

def stage7_outliers(df: pd.DataFrame,
                    cols: List[str] = ("revenue", "risk_index_var95"),
                    k_iqr: float = 1.5) -> OutlierResults:
    bounds = {}
    n_out = {}
    mask_ok = pd.Series(True, index=df.index)
    for c in cols:
        if c not in df.columns:
            continue
        lo, hi = iqr_bounds(df[c], k=k_iqr)
        bounds[c] = (lo, hi)
        out_mask = (df[c] < lo) | (df[c] > hi)
        n_out[c] = int(out_mask.sum())
        mask_ok &= ~out_mask

    df_no_out = df.loc[mask_ok].copy()
    df_wins = df.copy()
    for c in cols:
        if c in df_wins.columns:
            df_wins[c] = winsorize_series(df_wins[c])

    return OutlierResults(bounds, n_out, df_no_out, df_wins)

def check_assumptions_linear(y: pd.Series, x: pd.Series) -> Dict[str, float]:
    sxy = pd.DataFrame({"y": y, "x": x}).dropna()
    if len(sxy) < 3:
        return {"n": len(sxy)}
    return {
        "n": len(sxy),
        "corr_pearson": float(sxy["y"].corr(sxy["x"])),
        "var_y": float(np.nanvar(sxy["y"])),
        "var_x": float(np.nanvar(sxy["x"])),
    }

# ---------------------------
# Stage 8: EDA
# ---------------------------
@dataclass
class EDAResults:
    describe_numeric: pd.DataFrame
    missing_pct: pd.Series
    corr: pd.DataFrame

def stage8_eda(df: pd.DataFrame) -> EDAResults:
    num_df = df.select_dtypes(include=[np.number])
    desc = num_df.describe().T
    miss = df.isna().mean().sort_values(ascending=False) * 100.0
    corr = num_df.corr()
    return EDAResults(desc, miss, corr)

# ---------------------------
# Stage 9: Feature Engineering
# ---------------------------
def stage9_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Make sure rev_chg exists once
    if "revenue" in out.columns and "rev_chg" not in out.columns:
        out["rev_chg"] = out["revenue"].pct_change()

    if "risk_index_var95" in out.columns:
        out["risk_chg"] = out["risk_index_var95"].pct_change()
        out["risk_roll_std_3"] = out["risk_index_var95"].rolling(window=3, min_periods=2).std()
        out["risk_roll_std_5"] = out["risk_index_var95"].rolling(window=5, min_periods=2).std()

    if "revenue" in out.columns and "risk_index_var95" in out.columns:
        out["rev_x_risk"] = out["revenue"] * out["risk_index_var95"]
        out["revenue_lag1"] = out["revenue"].shift(1)
        out["risk_lag1"] = out["risk_index_var95"].shift(1)

    if "risk_index_var95" in out.columns:
        out["risk_up_next"] = (out["risk_index_var95"].shift(-1) > out["risk_index_var95"]).astype(float)

    return out

# ---------------------------
# Stage 10a: Linear Regression
# ---------------------------
@dataclass
class LRFit:
    coef: float
    intercept: float
    r2: float
    rmse: float
    n: int

def fit_lr_univariate(y: pd.Series, x: pd.Series) -> Optional[LRFit]:
    dfm = pd.DataFrame({"y": y, "x": x}).dropna()
    min_n = 3 if ALLOW_SMALL_N else 5
    if len(dfm) < min_n:
        return None
    X = dfm[["x"]].values
    Y = dfm["y"].values
    model = LinearRegression().fit(X, Y)
    pred = model.predict(X)
    return LRFit(
        coef=float(model.coef_[0]),
        intercept=float(model.intercept_),
        r2=float(r2_score(Y, pred)),
        rmse=rmse(Y, pred),
        n=len(dfm),
    )

@dataclass
class LRMultiFit:
    coefs: Dict[str, float]
    intercept: float
    r2: float
    rmse: float
    n: int
    features: List[str]

def fit_lr_multivariate(df: pd.DataFrame, y_col: Optional[str], x_cols: List[str]) -> Optional[LRMultiFit]:
    if (y_col is None) or (y_col not in df.columns):
        return None
    use_cols = [c for c in x_cols if c in df.columns]
    if not use_cols:
        return None
    dfm = df[[y_col] + use_cols].dropna()
    min_n = 5 if ALLOW_SMALL_N else 20  # still keep a little buffer for stability
    if len(dfm) < min_n:
        return None

    X = dfm[use_cols].values
    Y = dfm[y_col].values
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    model = LinearRegression().fit(Xs, Y)
    pred = model.predict(Xs)

    coefs = {f: float(w) for f, w in zip(use_cols, model.coef_)}
    return LRMultiFit(
        coefs=coefs,
        intercept=float(model.intercept_),
        r2=float(r2_score(Y, pred)),
        rmse=rmse(Y, pred),
        n=len(dfm),
        features=use_cols
    )

# ---------------------------
# Stage 10b: Time Series & Classification (guarded)
# ---------------------------
@dataclass
class TSResults:
    model: str
    aic: Optional[float]
    last_pred: Optional[float]

def stage10b_time_series(df: pd.DataFrame, y_col: str = "revenue") -> Optional[TSResults]:
    if (ARIMA is None) or (y_col not in df.columns):
        return None
    ser = pd.to_numeric(df[y_col], errors="coerce").dropna()
    if len(ser) < (25 if not ALLOW_SMALL_N else 12):
        return None
    try:
        model = ARIMA(ser, order=(1, 1, 1))
        fit = model.fit()
        fc = fit.forecast(steps=1)
        return TSResults(model="ARIMA(1,1,1)", aic=float(getattr(fit, "aic", np.nan)), last_pred=float(fc.iloc[-1]))
    except Exception:
        return None

@dataclass
class ClfResults:
    acc: float
    conf_mat: List[List[int]]
    n_train: int
    n_test: int
    features: List[str]

def stage10b_classification(df: pd.DataFrame) -> Optional[ClfResults]:
    target = "risk_up_next"
    base_feats = ["revenue", "risk_index_var95", "rev_chg", "risk_chg",
                  "risk_roll_std_3", "risk_roll_std_5", "rev_x_risk",
                  "revenue_lag1", "risk_lag1"]
    feats = [f for f in base_feats if f in df.columns]
    if (target not in df.columns) or not feats:
        return None

    d = df[feats + [target]].dropna()
    min_n = 80 if not ALLOW_SMALL_N else 20
    if len(d) < min_n:
        return None

    X = d[feats].values
    y = d[target].values

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)
    scaler = StandardScaler()
    X_trs = scaler.fit_transform(X_tr)
    X_tes = scaler.transform(X_te)

    clf = LogisticRegression(max_iter=200, solver="lbfgs")
    clf.fit(X_trs, y_tr)
    y_pred = clf.predict(X_tes)

    acc = float(accuracy_score(y_te, y_pred))
    cm = confusion_matrix(y_te, y_pred).tolist()
    return ClfResults(acc=acc, conf_mat=cm, n_train=len(y_tr), n_test=len(y_te), features=feats)

# ---------------------------
# Auto-pick latest files
# ---------------------------
def get_latest_file(folder="data/raw", exts=(".csv", ".parquet")) -> str:
    p = Path(folder)
    cands = [f for f in p.glob("*") if f.suffix.lower() in exts]
    if not cands:
        raise FileNotFoundError(f"No CSV/Parquet files found in {folder}")
    latest = max(cands, key=lambda f: f.stat().st_mtime)
    return str(latest)

def get_latest_api_for_ticker(ticker: str, folder="data/raw") -> Optional[str]:
    p = Path(folder)
    cands = list(p.glob(f"api_*_{ticker.upper()}*.csv")) + list(p.glob(f"api_*_{ticker.upper()}*.parquet"))
    if not cands:
        return None
    return str(max(cands, key=lambda f: f.stat().st_mtime))

# ---------------------------
# Orchestrator
# ---------------------------
def run_pipeline(path: str):
    print("="*78)
    print("LOAD")
    print("="*78)
    df = load_data(path)
    df = attach_risk_or_proxy(df)  # ensures risk_index_var95 exists (real or proxy)
    print(f"[LOAD] rows={len(df)} cols={list(df.columns)}")

    y_col = "revenue" if "revenue" in df.columns else None
    x_col = "risk_index_var95" if "risk_index_var95" in df.columns else None

    # Stage 7
    print("\n" + "="*78)
    print("STAGE 7: Outliers, Risk & Assumptions")
    print("="*78)
    out7 = stage7_outliers(df, cols=[c for c in [y_col, x_col] if c])
    print("[IQR bounds]:", out7.iqr_bounds)
    print("[n_outliers]:", out7.n_outliers)
    if y_col and x_col:
        asmp = check_assumptions_linear(df[y_col], df[x_col])
        print("[Assumptions Check]:", asmp)
    else:
        print("Skipping assumption check (missing revenue or risk_index_var95).")

    # Stage 8
    print("\n" + "="*78)
    print("STAGE 8: EDA")
    print("="*78)
    eda = stage8_eda(df)
    print("[Missing % Top 10]:")
    print(eda.missing_pct.head(10).round(2))
    print("\n[Numeric Describe Top 10]:")
    print(eda.describe_numeric.round(3).head(10))
    print("\n[Correlation (first 6x6)]:")
    print(eda.corr.round(3).iloc[:6, :6])

    # Stage 9
    print("\n" + "="*78)
    print("STAGE 9: Feature Engineering")
    print("="*78)
    df_feat_all = stage9_features(df)
    df_feat_noo = stage9_features(out7.df_no_outliers)
    df_feat_win = stage9_features(out7.df_winsorized)
    new_cols = [c for c in df_feat_all.columns if c not in df.columns]
    print("[Features] Added columns:", new_cols)

    # Stage 10a: LR (uni across treatments)
    print("\n" + "="*78)
    print("STAGE 10a: Linear Regression")
    print("="*78)
    if y_col and x_col:
        lr_all = fit_lr_univariate(df_feat_all[y_col], df_feat_all[x_col])
        print("\n[LR Univariate - ALL]:", vars(lr_all) if lr_all else "Insufficient data")

        lr_noo = fit_lr_univariate(df_feat_noo[y_col], df_feat_noo[x_col])
        print("[LR Univariate - NO OUTLIERS]:", vars(lr_noo) if lr_noo else "Insufficient data")

        lr_win = fit_lr_univariate(df_feat_win[y_col], df_feat_win[x_col])
        print("[LR Univariate - WINSORIZED]:", vars(lr_win) if lr_win else "Insufficient data")
    else:
        print("Skipping LR (missing revenue or risk_index_var95).")
        lr_all = lr_noo = lr_win = None

    mv_feats = ["risk_index_var95", "rev_chg", "risk_chg", "risk_roll_std_3",
                "risk_roll_std_5", "rev_x_risk", "revenue_lag1", "risk_lag1"]
    lr_multi = fit_lr_multivariate(df_feat_all, y_col="revenue" if "revenue" in df_feat_all.columns else None,
                                   x_cols=mv_feats)
    print("\n[LR Multivariate - Engineered Features]:", vars(lr_multi) if lr_multi else "Insufficient data")

 # --- Stage 10b: Classification (patched for tiny samples) ---
@dataclass
class ClfResults:
    acc: float
    conf_mat: List[List[int]]
    n_train: int
    n_test: int
    features: List[str]

def stage10b_classification(df: pd.DataFrame) -> Optional[ClfResults]:
    target = "risk_up_next"
    base_feats = ["revenue", "risk_index_var95", "rev_chg", "risk_chg",
                  "risk_roll_std_3", "risk_roll_std_5", "rev_x_risk",
                  "revenue_lag1", "risk_lag1"]
    feats = [f for f in base_feats if f in df.columns]
    if (target not in df.columns) or not feats:
        return None

    d = df[feats + [target]].dropna()
    # allow as few as 5 rows in small-N mode (demo only)
    min_n = 5 if ALLOW_SMALL_N else 80
    if len(d) < min_n:
        return None

    # must have both classes present for a meaningful classifier
    if d[target].nunique() < 2:
        return None

    X = d[feats].values
    y = d[target].values

    # for very small N, use a bigger test split to ensure at least 1 test point
    test_size = 0.4 if len(d) <= 10 else 0.2
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, shuffle=True, random_state=42)

    # safety: if the train set collapsed to a single class, bail out
    if len(np.unique(y_tr)) < 2:
        return None

    scaler = StandardScaler()
    X_trs = scaler.fit_transform(X_tr)
    X_tes = scaler.transform(X_te)

    clf = LogisticRegression(max_iter=200, solver="lbfgs")
    clf.fit(X_trs, y_tr)
    y_pred = clf.predict(X_tes)

    acc = float(accuracy_score(y_te, y_pred))
    cm = confusion_matrix(y_te, y_pred, labels=[0,1]).tolist()
    return ClfResults(acc=acc, conf_mat=cm, n_train=len(y_tr), n_test=len(y_te), features=feats)


# ---------------------------
# Entry point
# ---------------------------
if __name__ == "__main__":
    chosen = None
    if TICKER_PREFERENCE:
        chosen = get_latest_api_for_ticker(TICKER_PREFERENCE, "data/raw")
        if chosen:
            print(f"[AUTO] Using latest {TICKER_PREFERENCE} API file: {chosen}")
        else:
            print(f"[WARN] No API files found for ticker {TICKER_PREFERENCE} in data/raw/. Falling back to any latest file.")

    if not chosen:
        chosen = get_latest_file("data/raw")
        print(f"[AUTO] Using latest file: {chosen}")

    run_pipeline(chosen)




[AUTO] Using latest JPM API file: data\raw\api_yf_JPM_20250828-1740.csv
LOAD
[LOAD] rows=5 cols=['date', 'revenue', 'source', 'ticker', 'risk_index_var95']

STAGE 7: Outliers, Risk & Assumptions
[IQR bounds]: {'revenue': (np.float64(39317000000.0), np.float64(48221000000.0)), 'risk_index_var95': (np.float64(-10.40873778155948), np.float64(184.9321168599255))}
[n_outliers]: {'revenue': 0, 'risk_index_var95': 0}
[Assumptions Check]: {'n': 3, 'corr_pearson': 0.9225690512029692, 'var_y': 1.2224002222222223e+18, 'var_x': 1734.153887410438}

STAGE 8: EDA
[Missing % Top 10]:
risk_index_var95    40.0
date                 0.0
revenue              0.0
source               0.0
ticker               0.0
dtype: float64

[Numeric Describe Top 10]:
                  count          mean           std           min  \
revenue             5.0  4.354480e+10  1.458035e+09  4.206800e+10   
risk_index_var95    3.0  8.301600e+01  5.100200e+01  2.568800e+01   

                           25%           50%     